# RMSNorm

源码导航：[`RMSNorm`](../../../core/norm/rmsnorm.py#L16)。

RMSNorm 解决的是“只需要控制激活尺度，不一定要强制零均值”的归一化问题。对最后一维 $d$ 做均方根归一化：

$$
\operatorname{RMSNorm}(x)=\frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon}}\odot w
$$

与 LayerNorm 相比，它去掉了减均值与 bias，参数更少、计算路径更短。Walkie 的实现会先把输入转到 fp32 计算 RMS，再回到原 dtype，这对 bf16/fp16 训练更稳。改进思路是把 GPT-2 风格的 LayerNorm 换成现代 decoder-only 模型常用的 RMSNorm，让大模型的 pre-norm 残差流更轻。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torch.nn as nn

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.rmsnorm import RMSNorm

## 1. 形状与 RMS 检查

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 4, 16) * 3
norm = RMSNorm(16, eps=1e-6)
y = norm(x)

print('x.shape =', tuple(x.shape))
print('y.shape =', tuple(y.shape))
print('RMS before:', x.pow(2).mean(dim=-1).sqrt().mean().item())
print('RMS after :', y.pow(2).mean(dim=-1).sqrt().mean().item())

## 2. 与 LayerNorm 的参数量对比

In [ ]:
dim = 1536
rms = RMSNorm(dim)
ln = nn.LayerNorm(dim, elementwise_affine=True)
print('RMSNorm params:', sum(p.numel() for p in rms.parameters()))
print('LayerNorm params:', sum(p.numel() for p in ln.parameters()))

---

## 延伸阅读与参考资料

### 核心论文
- **Root Mean Square Layer Normalization**: Zhang and Sennrich, 2019. [arXiv:1910.07467](https://arxiv.org/abs/1910.07467)
- **T5**: Raffel et al., 2019. [arXiv:1910.10683](https://arxiv.org/abs/1910.10683)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)

### 工程实现
- **Hugging Face Transformers LlamaRMSNorm**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)
- **PyTorch LayerNorm API**: [docs](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)